# Build a risk-analyst LangChain agent in 25 lines

**Problem:** LLMs are unreliable at *computing* finance math. Ask one to price a Black-Scholes call and the answer drifts run to run. Ask for the higher-order Greeks and they're frequently wrong.

**Fix:** keep the LLM for reasoning, hand the math to a deterministic API.

This notebook shows a LangChain agent that answers risk and portfolio questions using [QuantOracle](https://api.quantoracle.dev) — 63 deterministic quant calculators + 10 composite workflows, all citation-verified against Hull / Wilmott / Bailey & Lopez de Prado.

- 1,000 free calls per IP per day, no signup, no API key
- Paid composites via [x402](https://x402.org) USDC micropayments on Base or Solana
- Cataloged in [CDP Bazaar](https://api.cdp.coinbase.com/platform/v2/x402/discovery/resources) for agent discovery

## 1. Install

In [1]:
%pip install --quiet langchain-quantoracle langchain-openai langgraph

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: D:\Quantcalc\venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import os
# Set OPENAI_API_KEY in your environment, or uncomment + paste below:
# os.environ["OPENAI_API_KEY"] = "sk-..."
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY first"

## 2. Load just the risk + portfolio + stats tools

QuantOracle exposes 73 endpoints. Loading all of them into an agent prompt confuses smaller models. Filter by category to keep the tool list focused.

In [3]:
from langchain_quantoracle import QuantOracleToolkit

tools = QuantOracleToolkit(categories=["risk", "portfolio", "stats"]).get_tools()
print(f"Loaded {len(tools)} tools")
for t in tools[:5]:
    print(f"  • {t.name}")
print("  ...")

D:\Quantcalc\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Loaded 25 tools
  • risk_portfolio
  • risk_kelly
  • risk_correlation
  • risk_position_size
  • risk_drawdown
  ...


## 3. Build the agent

In [4]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

SYSTEM = (
    "You are a quantitative analyst. For ANY financial calculation — Sharpe, "
    "VaR, drawdown, Kelly, position sizing, regression — you MUST call a "
    "QuantOracle tool. Never compute math in your head. After tools return, "
    "explain the result in plain English and recommend an action."
)

agent = create_react_agent(llm, tools, prompt=SYSTEM)

C:\Users\sfelc\AppData\Local\Temp\ipykernel_22592\1686872763.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=SYSTEM)


## 4. Ask a risk question with real returns

These are 20 days of made-up daily returns. The agent should pick `risk_portfolio` (a free tool that bundles Sharpe, Sortino, Calmar, VaR, CVaR, drawdown, win rate, fat-tail check) and contextualize the result.

In [5]:
returns = [
    0.012, -0.008, 0.015, -0.022, 0.018, 0.005, -0.011, 0.014, -0.003, 0.009,
    -0.018, 0.020, -0.007, 0.012, 0.003, -0.013, 0.016, -0.005, 0.011, -0.009,
]

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            f"Here are my last 20 daily returns: {returns}. "
            "Am I taking too much risk for the return I'm getting? "
            "Give me Sharpe, max drawdown, and one sentence of advice."
        ),
    }]
})

print(result["messages"][-1].content)

Your Sharpe ratio is approximately **2.15**, which indicates that you are generating a good return relative to the risk you are taking. The maximum drawdown is **-2.1%**, meaning that at one point, your portfolio value dropped by this percentage from its peak.

**Advice:** Your risk-adjusted returns are strong, but keep an eye on the drawdown; consider setting stop-loss orders to manage potential losses more effectively.


Behind the scenes that one call returned (real API output, deterministic):

```json
{
  "returns": {"annualized": 0.4914, "vol": 0.2052, "win_rate": 0.55, "profit_factor": 1.4062},
  "risk":    {"sharpe": 2.151, "sortino": 3.3664, "calmar": 22.3364, "max_drawdown": -0.022,
              "var_95": -0.022, "cvar_95": -0.022},
  "distribution": {"skewness": -0.2911, "excess_kurtosis": -1.2361, "fat_tails": false},
  "n": 20, "ms": 18.5
}
```

Same input → same output, every time. The LLM doesn't *do* the math — it interprets the result.

## 5. Position sizing with Kelly criterion

In [6]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "I have a strategy with a 55% win rate. Average winner: $1.50. "
            "Average loser: $1.00. What fraction of my account should each "
            "trade risk? Give me full Kelly, half-Kelly, and quarter-Kelly."
        ),
    }]
})

print(result["messages"][-1].content)

Based on your strategy with a 55% win rate, an average winner of $1.50, and an average loser of $1.00, here are the recommended fractions of your account to risk per trade:

1. **Full Kelly**: 25% of your account
2. **Half Kelly**: 12.5% of your account
3. **Quarter Kelly**: 6.25% of your account

### Explanation:
- **Full Kelly** is the maximum amount you can risk based on your edge and payoff ratio. In this case, it suggests risking 25% of your account on each trade.
- **Half Kelly** is a more conservative approach, suggesting you risk 12.5% of your account, which can help mitigate risk while still taking advantage of your strategy's edge.
- **Quarter Kelly** is the most conservative option, recommending a risk of 6.25% of your account. This is often recommended for those who want to minimize volatility and drawdowns.

### Recommendation:
I recommend using the **Quarter Kelly** strategy, risking 6.25% of your account per trade. This approach balances risk and reward, allowing you to 

## 6. Composite workflows: when one tool replaces ten

QuantOracle has 10 **composite** endpoints that bundle 5–15 calculator calls into a single round trip with a purpose-built output. They're paid-only via x402 (USDC on Base or Solana).

Examples:
- `backtest_strategy` — full SMA / RSI / momentum backtest with equity curve + buy-hold benchmark ($0.10)
- `portfolio_rebalance-plan` — generate the trade list to reach target weights with cost estimate ($0.05)
- `options_strategy-optimizer` — rank top option strategies given your outlook + vol view ($0.08)
- `hedging_recommend` — compare protective puts, collars, inverse hedges ($0.04)
- `risk_full-analysis` — full tearsheet from a returns array ($0.04)

If you're using an x402-capable HTTP client (e.g. [AgentCash](https://agentcash.dev), Coinbase AgentKit), payment is automatic — your agent gets a 402, signs an EIP-3009 USDC transfer, and retries. Otherwise the toolkit raises an `HTTPError` and you can layer your own payment flow.

```python
# Example — same pattern, just include the composite categories
tools = QuantOracleToolkit(
    categories=["risk", "backtest", "portfolio", "hedging"]
).get_tools()
```

## What this pattern buys you

| | LLM-only | LLM + QuantOracle |
|---|---|---|
| Reproducibility | drifts run-to-run | deterministic |
| Higher-order Greeks (vanna, charm, speed) | frequently wrong | exact |
| Iterative numerics (IV solver, GARCH, optimizer) | degrades | converges |
| Cost per call | LLM tokens (~$0.02–0.10) | $0 free tier, then $0.002–$0.10 |
| Audit trail | none | every call hashable + replayable |

When an agent makes 50 tool calls during a backtest, every calculation has to be right. The pattern — **LLM for reasoning, deterministic API for compute** — is what actually works in production.

## Links

- API + docs: <https://api.quantoracle.dev/docs>
- Tool catalog: <https://api.quantoracle.dev/tools>
- GitHub: <https://github.com/QuantOracledev/quantoracle>
- PyPI: [`langchain-quantoracle`](https://pypi.org/project/langchain-quantoracle/)
- MCP server: `npx quantoracle-mcp` ([npm](https://www.npmjs.com/package/quantoracle-mcp))
- CDP Bazaar listings: <https://api.cdp.coinbase.com/platform/v2/x402/discovery/resources> (filter `resource` for `quantoracle.dev`)
- x402 discovery feed: <https://api.quantoracle.dev/.well-known/x402>